In [ ]:
from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

DATA_DIR = Path(
    "/blue/bala1s/krishnap.kalivel/"
    "TLFHydrodynamics/thinfilm-closure-ml/"
    "datasets/BasC/param_space/ClosureData"
)

FILE_PATTERN = "thinfilm_closure_data_*.h5"

# Protect ratios against division by extremely small q.
# Set to 0.0 if q is guaranteed to remain safely away from zero.
Q_ABS_MIN = 1e-10

files = sorted(DATA_DIR.glob(FILE_PATTERN))

print(f"Found {len(files)} HDF5 files.")

for f in files[:10]:
    print(f.name)

if len(files) > 10:
    print(f"... and {len(files) - 10} more")

In [ ]:
def parse_case_name(filename):
    """
    Parse Re, Ca, theta from filenames like:

    thinfilm_closure_data_Re010_Ca0_100_theta45.h5
    thinfilm_closure_data_Re120_Ca0_010_theta6_4.h5
    """

    name = Path(filename).stem

    pattern = (
        r"Re(?P<Re>\d+)_"
        r"Ca(?P<Ca>\d+_\d+)_"
        r"theta(?P<theta>\d+(?:_\d+)?)"
    )

    match = re.search(pattern, name)

    if match is None:
        return {
            "Re": np.nan,
            "Ca": np.nan,
            "theta": np.nan
        }

    Re = float(match.group("Re"))

    Ca_string = match.group("Ca").replace("_", ".")
    Ca = float(Ca_string)

    theta_string = match.group("theta").replace("_", ".")
    theta = float(theta_string)

    return {
        "Re": Re,
        "Ca": Ca,
        "theta": theta
    }


# Test
for f in files[:5]:
    print(f.name, "->", parse_case_name(f.name))

In [ ]:
# ---------------------------------------------------------
# Mask configuration
# ---------------------------------------------------------

H_MEAN = 1.0
H_TOLERANCE = 0.1
OUTLET_EXCLUDE_FRACTION = 0.05

ONLY_DISTURBED_POINTS = False


def load_closure_quantities(filepath, q_abs_min=1e-10):
    """
    Apply:
      1. Film-disturbance mask at each time.
      2. Exclusion of the final 5% of the spatial domain.
      3. Finite-value, positive-height, and nonzero-flow checks.

    Supports fields stored as (nx, nt) or (nt, nx), using x and t
    to identify the spatial axis.

    Returns flattened valid closure quantities.
    """

    with h5py.File(filepath, "r") as f:
        x = np.asarray(f["x"][:], dtype=float).reshape(-1)
        t = np.asarray(f["t"][:], dtype=float).reshape(-1)

        fields = {
            name: np.asarray(f[name][:], dtype=float)
            for name in [
                "h", "q", "shape_factor", "u_interface", "tau_wall"
            ]
        }

    h_shape = fields["h"].shape

    if any(field.shape != h_shape for field in fields.values()):
        raise ValueError(f"{filepath.name}: field shapes do not match.")

    if len(h_shape) != 2:
        raise ValueError(
            f"{filepath.name}: expected 2D fields, got {h_shape}."
        )

    # Convert all fields to (nx, nt).
    if x.size == t.size and h_shape == (x.size, t.size):
        raise ValueError(
            f"{filepath.name}: spatial axis is ambiguous because nx == nt."
        )
    elif h_shape == (x.size, t.size):
        pass
    elif h_shape == (t.size, x.size):
        fields = {name: field.T for name, field in fields.items()}
    else:
        raise ValueError(
            f"{filepath.name}: field shape {h_shape} does not match "
            f"nx={x.size}, nt={t.size}."
        )

    if x.size < 2 or not np.all(np.isfinite(x)) or np.any(np.diff(x) <= 0):
        raise ValueError(
            f"{filepath.name}: x must be finite and strictly increasing."
        )

    h = fields["h"]
    q = fields["q"]
    shape = fields["shape_factor"]
    ui = fields["u_interface"]
    tau = fields["tau_wall"]

    # -----------------------------------------------------
    # Exclude the final 5% by spatial coordinate
    # -----------------------------------------------------

    x_cutoff = (
        x[0]
        + (1.0 - OUTLET_EXCLUDE_FRACTION) * (x[-1] - x[0])
    )
    spatial_mask = x <= x_cutoff

    # Identify disturbances only within the retained domain.
    # Invalid heights are not treated as disturbances.
    disturbed = (
        np.isfinite(h)
        & (np.abs(h - H_MEAN) > H_TOLERANCE)
        & spatial_mask[:, None]
    )

    # -----------------------------------------------------
    # Film-disturbance mask
    # -----------------------------------------------------

    analysis_mask = np.zeros_like(h, dtype=bool)

    if ONLY_DISTURBED_POINTS:
        analysis_mask = disturbed.copy()

    else:
        for j in range(h.shape[1]):
            disturbed_indices = np.flatnonzero(disturbed[:, j])

            if disturbed_indices.size:
                last_disturbed_index = disturbed_indices[-1]
                analysis_mask[:last_disturbed_index + 1, j] = True

    analysis_mask &= spatial_mask[:, None]

    # -----------------------------------------------------
    # Numerical validity checks
    # -----------------------------------------------------

    valid = (
        analysis_mask
        & np.isfinite(h)
        & np.isfinite(q)
        & np.isfinite(shape)
        & np.isfinite(ui)
        & np.isfinite(tau)
        & (h > 0.0)
        & (np.abs(q) > q_abs_min)
    )

    phi = np.full_like(h, np.nan)
    ui_ratio = np.full_like(h, np.nan)

    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        phi[valid] = shape[valid] / (q[valid]**2 / h[valid])
        ui_ratio[valid] = ui[valid] / (q[valid] / h[valid])

    final_valid = (
        valid
        & np.isfinite(phi)
        & np.isfinite(ui_ratio)
    )

    n_total = h.size
    n_masked = int(np.count_nonzero(analysis_mask))
    n_valid = int(np.count_nonzero(final_valid))

    print(
        f"  Mask retained: {100 * n_masked / n_total:.2f}%"
        f" | Valid closure data: {100 * n_valid / n_total:.2f}%"
    )

    return {
        "phi": phi[final_valid].ravel(),
        "ui_ratio": ui_ratio[final_valid].ravel(),
        "tau_wall": tau[final_valid].ravel(),
        "n_total": n_total,
        "n_masked": n_masked,
        "n_valid": n_valid,
    }

In [ ]:
def compute_stats(x):
    """
    Calculate useful distribution statistics.
    """

    x = np.asarray(x)
    x = x[np.isfinite(x)]

    if x.size == 0:
        return {
            "min": np.nan,
            "p001": np.nan,
            "p01": np.nan,
            "p1": np.nan,
            "p5": np.nan,
            "median": np.nan,
            "p95": np.nan,
            "p99": np.nan,
            "p999": np.nan,
            "max": np.nan,
            "mean": np.nan,
            "std": np.nan,
        }

    percentiles = np.percentile(
        x,
        [0.1, 1, 5, 50, 95, 99, 99.9]
    )

    return {
        "min": np.min(x),
        "p001": percentiles[0],
        "p01": percentiles[1],
        "p1": percentiles[2],
        "median": percentiles[3],
        "p95": percentiles[4],
        "p99": percentiles[5],
        "p999": percentiles[6],
        "max": np.max(x),
        "mean": np.mean(x),
        "std": np.std(x),
    }

In [ ]:
rows = []

for i, filepath in enumerate(files, start=1):

    case = parse_case_name(filepath.name)

    print(
        f"[{i:3d}/{len(files):3d}] "
        f"Re={case['Re']:g}, "
        f"Ca={case['Ca']:g}, "
        f"theta={case['theta']:g}"
    )

    data = load_closure_quantities(
        filepath,
        q_abs_min=Q_ABS_MIN
    )

    for quantity in ["phi", "ui_ratio", "tau_wall"]:

        stats = compute_stats(data[quantity])

        row = {
            "file": filepath.name,
            "Re": case["Re"],
            "Ca": case["Ca"],
            "theta": case["theta"],
            "quantity": quantity,
            "n_total": data["n_total"],
            "n_valid": data["n_valid"],
            "valid_fraction": data["n_valid"] / data["n_total"],
        }

        row.update(stats)

        rows.append(row)


stats_df = pd.DataFrame(rows)

print("\nDone.")
print(f"Processed {len(files)} files.")

In [ ]:
stats_df

In [ ]:
display_cols = [
    "Re",
    "Ca",
    "theta",
    "quantity",
    "min",
    "p01",
    "median",
    "p99",
    "max",
    "mean",
    "std",
    "valid_fraction"
]

stats_df[display_cols]

In [ ]:
phi_df = stats_df[
    stats_df["quantity"] == "phi"
].reset_index(drop=True)

ui_df = stats_df[
    stats_df["quantity"] == "ui_ratio"
].reset_index(drop=True)

tau_df = stats_df[
    stats_df["quantity"] == "tau_wall"
].reset_index(drop=True)

In [ ]:
phi_df[
    ["Re", "Ca", "theta", "min", "p01", "median", "p99", "max"]
]

In [ ]:
ui_df[
    ["Re", "Ca", "theta", "min", "p01", "median", "p99", "max"]
]

In [ ]:
tau_df[
    ["Re", "Ca", "theta", "min", "p01", "median", "p99", "max"]
]

In [ ]:
for quantity in ["phi", "ui_ratio", "tau_wall"]:

    dfq = stats_df[stats_df["quantity"] == quantity]

    min_row = dfq.loc[dfq["min"].idxmin()]
    max_row = dfq.loc[dfq["max"].idxmax()]

    print("=" * 70)
    print(quantity)
    print("=" * 70)

    print("\nGlobal minimum:")
    print(
        f"  value = {min_row['min']:.6g}\n"
        f"  Re    = {min_row['Re']:g}\n"
        f"  Ca    = {min_row['Ca']:g}\n"
        f"  theta = {min_row['theta']:g}\n"
        f"  file  = {min_row['file']}"
    )

    print("\nGlobal maximum:")
    print(
        f"  value = {max_row['max']:.6g}\n"
        f"  Re    = {max_row['Re']:g}\n"
        f"  Ca    = {max_row['Ca']:g}\n"
        f"  theta = {max_row['theta']:g}\n"
        f"  file  = {max_row['file']}"
    )

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Configuration — choose your simulation here
# ---------------------------------------------------------

DATA_DIR = Path(
    "/blue/bala1s/krishnap.kalivel/"
    "TLFHydrodynamics/thinfilm-closure-ml/"
    "datasets/BasC/param_space/ClosureData"
)

RE = 40
CA = 0.01
THETA = 90       # Also supports values such as 6.4

re_tag = f"{RE:03d}"
ca_tag = f"{CA:.3f}".replace(".", "_")
theta_tag = f"{THETA:g}".replace(".", "_")

SELECTED_FILE = (
    f"thinfilm_closure_data_"
    f"Re{re_tag}_Ca{ca_tag}_theta{theta_tag}.h5"
)

H_MEAN = 1.0
H_TOLERANCE = 0.1
OUTLET_EXCLUDE_FRACTION = 0.05
Q_ABS_MIN = 1e-10

filepath = DATA_DIR / SELECTED_FILE

# ---------------------------------------------------------
# Load the selected simulation
# ---------------------------------------------------------

with h5py.File(filepath, "r") as f:
    x = np.asarray(f["x"][:], dtype=float).squeeze().reshape(-1)
    t = np.asarray(f["t"][:], dtype=float).squeeze().reshape(-1)

    h = np.asarray(f["h"][:], dtype=float)
    q = np.asarray(f["q"][:], dtype=float)
    momentum_integral = np.asarray(f["shape_factor"][:], dtype=float)

# Standardize arrays to shape (nx, nt).
if not (h.shape == q.shape == momentum_integral.shape):
    raise ValueError("h, q, and shape_factor must have matching shapes.")

if h.ndim != 2:
    raise ValueError(f"Expected 2D fields; received shape {h.shape}.")

if x.size == t.size and h.shape == (x.size, t.size):
    raise ValueError("Cannot determine spatial axis because nx == nt.")
elif h.shape == (x.size, t.size):
    pass
elif h.shape == (t.size, x.size):
    h = h.T
    q = q.T
    momentum_integral = momentum_integral.T
else:
    raise ValueError(
        f"Field shape {h.shape} does not match "
        f"nx={x.size}, nt={t.size}."
    )

if (
    x.size < 2
    or not np.all(np.isfinite(x))
    or np.any(np.diff(x) <= 0)
):
    raise ValueError("x must be finite and strictly increasing.")

# ---------------------------------------------------------
# Exclude the final 5% of the spatial domain
# ---------------------------------------------------------

x_cutoff = (
    x[0]
    + (1.0 - OUTLET_EXCLUDE_FRACTION) * (x[-1] - x[0])
)
spatial_mask = x <= x_cutoff

# ---------------------------------------------------------
# Your film mask:
# retain points through the last disturbed point at each time
# ---------------------------------------------------------

disturbed = (
    np.isfinite(h)
    & (np.abs(h - H_MEAN) > H_TOLERANCE)
    & spatial_mask[:, None]
)

analysis_mask = np.zeros_like(h, dtype=bool)

for j in range(t.size):
    disturbed_indices = np.flatnonzero(disturbed[:, j])

    if disturbed_indices.size:
        last_disturbed_index = disturbed_indices[-1]
        analysis_mask[:last_disturbed_index + 1, j] = True

analysis_mask &= spatial_mask[:, None]

# ---------------------------------------------------------
# Calculate phi using valid retained samples
#
# shape_factor is assumed to store the momentum integral:
#     integral_0^h u² dy
# ---------------------------------------------------------

valid = (
    analysis_mask
    & np.isfinite(h)
    & np.isfinite(q)
    & np.isfinite(momentum_integral)
    & (h > 0.0)
    & (np.abs(q) > Q_ABS_MIN)
)

phi = np.full(h.shape, np.nan, dtype=float)

with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    phi[valid] = (
        momentum_integral[valid] / (q[valid]**2 / h[valid])
    )

phi[~np.isfinite(phi)] = np.nan

if not np.any(np.isfinite(phi)):
    raise ValueError("No valid phi values remain after masking.")

# ---------------------------------------------------------
# Find maximum phi over all retained positions and times
# ---------------------------------------------------------

ix_max, it_max = np.unravel_index(
    np.nanargmax(phi), phi.shape
)

phi_max = phi[ix_max, it_max]
x_max = x[ix_max]
t_max = t[it_max]
h_at_max = h[ix_max, it_max]

print(f"Simulation: {filepath.name}")
print(f"Maximum phi: {phi_max:.8g}")
print(f"Time:        {t_max:.8g}  (index {it_max})")
print(f"Position:    {x_max:.8g}  (index {ix_max})")
print(f"h:           {h_at_max:.8g}")
print(f"q:           {q[ix_max, it_max]:.8g}")
print(f"Valid samples retained: {np.isfinite(phi).mean():.2%}")



In [ ]:
# ---------------------------------------------------------
# Plot h, phi, and phi*q²/h at the time of maximum phi
# ---------------------------------------------------------

momentum_from_phi = np.full(h.shape[0], np.nan)

valid_snapshot = np.isfinite(phi[:, it_max])

momentum_from_phi[valid_snapshot] = (
    phi[valid_snapshot, it_max]
    * q[valid_snapshot, it_max]**2
    / h[valid_snapshot, it_max]
)

fig, axes = plt.subplots(
    3, 1, figsize=(12, 9), sharex=True
)

profiles = [
    h[:, it_max],
    phi[:, it_max],
    momentum_from_phi,
]

labels = [
    r"$h(x,t)$",
    r"$\phi$",
    r"$\phi q^2/h = \int_0^h u^2\,dy$",
]

colors = ["tab:blue", "tab:orange", "tab:green"]

for ax, profile, label, color in zip(
    axes, profiles, labels, colors
):
    ax.plot(x, profile, color=color)

    ax.scatter(
        x_max, profile[ix_max],
        color="tab:red", s=65, zorder=5,
        label=r"Location of maximum $\phi$"
    )

    ax.axvline(
        x_max, color="tab:red", linestyle="--", alpha=0.7
    )

    ax.axvspan(
        x_cutoff, x[-1], color="gray", alpha=0.15
    )

    ax.set_ylabel(label)
    ax.grid(alpha=0.25)

axes[0].set_title(f"{filepath.stem}\nt = {t_max:.6g}")
axes[0].legend()

axes[1].annotate(
    rf"$\phi_{{\max}}={phi_max:.5g}$" + f"\nx = {x_max:.5g}",
    xy=(x_max, phi_max),
    xytext=(12, -35),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color="tab:red")
)

axes[-1].set_xlabel("x")
axes[-1].set_xlim(x[0], x[-1])

axes[1].set_ylim(0.5, 2.0)
plt.tight_layout()
plt.show()

In [ ]:
if not np.any(np.isfinite(ui_normalized)):
    raise ValueError("No valid normalized interface velocities remain.")

ix_max, it_max = np.unravel_index(
    np.nanargmax(ui_normalized),
    ui_normalized.shape
)

ui_normalized_max = ui_normalized[ix_max, it_max]
x_max = x[ix_max]
t_max = t[it_max]

print(f"Maximum normalized u_i = {ui_normalized_max:.8g}")
print(f"Time = {t_max:.8g}, position = {x_max:.8g}")

# Recompute profiles at the newly selected time.
ui_profile = np.where(
    analysis_mask[:, it_max] & np.isfinite(ui[:, it_max]),
    ui[:, it_max],
    np.nan
)

tau_profile = np.where(
    analysis_mask[:, it_max] & np.isfinite(tau[:, it_max]),
    tau[:, it_max],
    np.nan
)
# ---------------------------------------------------------
# Plot interface, u_i, normalized u_i, and tau_w
# ---------------------------------------------------------

fig, axes_ui = plt.subplots(
    3, 1, figsize=(12, 9), sharex=True
)

profiles = [
    h[:, it_max],
    ui_profile,
    ui_normalized[:, it_max],
    tau_profile,
]

labels = [
    r"$h(x,t)$",
    r"$u_i$",
    r"$u_i/(q/h)$",
]

colors = ["tab:blue", "tab:green", "tab:orange", "tab:purple"]

for ax, profile, label, color in zip(
    axes_ui, profiles, labels, colors
):
    ax.plot(x, profile, color=color)

    if np.isfinite(profile[ix_max]):
        ax.scatter(
            x_max, profile[ix_max],
            color="tab:red", s=65, zorder=5
        )

    ax.axvline(
        x_max, color="tab:red", linestyle="--", alpha=0.7,
        label=r"Location of maximum $u_i/(q/h)$"
    )

    ax.axvspan(
        x_cutoff, x[-1], color="gray", alpha=0.15
    )

    ax.set_ylabel(label)
    ax.grid(alpha=0.25)

axes_ui[0].set_title(f"{filepath.stem}\nt = {t_max:.6g}")
axes_ui[0].legend()

axes_ui[-1].set_xlabel("x")
axes_ui[-1].set_xlim(x[0], x[-1])

axes_ui[2].set_ylim(0, 2)
plt.tight_layout()
plt.show()